# Data cleaning before training 

Data quality summary:

1000-level total reviews: 2189

2000-level total reviews: 871

3000-level total reviews: 424

4000-level total reviews: 316

5000-level total reviews: 104


1000-level course with comment count: 14

2000-level course with comment count: 8

3000-level course with comment count: 13

4000-level course with comment count: 37

5000-level course with comment count: 6

Path：\IEDA3560_Project\ust_reviews\{course_name}\{course_name}_reviews.csv

feature number= column number: 30

feature names: ['hash', 'semester', 'instructors', 'is_author', 'author', 'date', 'title', 'comment_content', 'comment_teaching', 'comment_grading', 'comment_workload', 'rating_content', 'rating_teaching', 'rating_grading', 'rating_workload', 'has_midterm', 'has_final', 'has_quiz', 'has_assignment', 'has_essay', 'has_project', 'has_attendance', 'has_reading', 'has_presentation', 'upvote_count', 'vote_count', 'voted', 'is_upvote', 'comment_count', 'attachments']


quality:
A lot of binary variable, with integer, the comments are the mixture of Eng and Chin, with oral expression 
Some unique key can be dropped 


Training:

We will use random forest regression or classification as the baseline 

Second iteration: add more feature to evaluate the performance 


# Baseline Data Cleaning Rules:

For each course csv file: in path \IEDA3560_Project\ust_reviews
1. Add the course_name as the tag for each csv fill the column with the column name on its path
2. Add the column: level,  for the fifth digit of the column course name if digit 5 =1 ==> fill:1000  ==2 ==> fill 2000 eg COMP1021 ==> fill 1000
3. Add a column: upvote_ratio, value for each cell  = upvote_count/ vote_count
4. Add a column: insturctor_name, value = get the first json in column instructors, key = name  which is the course insturtor name, we will use it to map professor rating from the raw data from
https://ust-rankings.com/ ,

the cleaned data path \IEDA3560_Project\ust_reviews\instructor_ranking.csv

if the value in column insturctors = [], drop this row. recored the number of review dropped due to the empty insturctors value

and we create a new column insturctor_rating to store the value after mapping the instructor name and its rating ## to prevent the data leakage, the calculation for the score is not the same as those in ustranking website, as we eliminated the workload rating. 

=  A+ =11 A=10 A-=9 B+=8, B=7, B-=6, C+=5, C=4, C-=3, D=2, E=1, F=0 

5. sort the data with date, only keeping the data within recent 5 years e.g. date = xxx xx, year, year= range[2020,2026]
6. for each course keep at most 50 recent reviews
7. save a df_course_name with column  [ 'rating_workload','rating_content', 'rating_teaching', 'rating_grading','has_midterm', 'has_final', 'has_quiz', 'has_assignment', 'has_essay', 'has_project', 'has_attendance', 'has_reading', 'has_presentation','course_name,level,'upvote_ratio','insturctor_rating']

 equipvalent to dropping ['hash', 'semester', 'instructors', 'is_author', 'author', 'date', 'title', 'comment_content', 'comment_teaching', 'comment_grading', 'comment_workload', 'upvote_count', 'vote_count', 'voted', 'is_upvote', 'comment_count', 'attachments', insturctor_name]

8. check the data remaining and iterate e.g. check how many value counted in total, if it filter out too much, then adjust the year range 


In [29]:
from pathlib import Path
import ast
import numpy as np
import pandas as pd


In [74]:
# mapping function to define instructor score based on ranking
def build_instructor_score_map(ranking_path):
    rating_map = {
        "A+": 11, "A": 10, "A-": 9,
        "B+": 8, "B": 7, "B-": 6,
        "C+": 5, "C": 4, "C-": 3,
        "D": 2, "E": 1, "F": 0
    }

    rank_df = pd.read_csv(ranking_path, encoding="utf-8-sig")
    rank_df.columns = [c.strip() for c in rank_df.columns]

    if "instructor name" not in rank_df.columns or "ranking" not in rank_df.columns:
        raise ValueError("instructor_ranking.csv must contain: instructor name, ranking")

    rank_df["instructor name"] = rank_df["instructor name"].astype(str).str.strip()
    rank_df["ranking"] = rank_df["ranking"].astype(str).str.strip().str.upper()
    rank_df["insturctor_rating"] = rank_df["ranking"].map(rating_map)

    name_to_score = (
        rank_df.dropna(subset=["instructor name"])
               .drop_duplicates(subset=["instructor name"], keep="first")
               .set_index("instructor name")["insturctor_rating"]
               .to_dict()
    )
    return name_to_score, rank_df

RANKING_PATH = ROOT / "instructor_ranking.csv"
name_to_score, rank_df = build_instructor_score_map(RANKING_PATH)
display(rank_df.head())

,instructor name,ranking,score,insturctor_rating
0,"TSOI, Yau Chat",A+,0.711526,11
1,"FONG, Tsz Ho",A+,0.567629,11
2,"NG, Ka Man",A+,0.528440,11
3,"BAO, Zhigang",A+,0.525558,11
4,"ARYA, Sunil",A+,0.514835,11


In [75]:
name_to_score

{'TSOI, Yau Chat': 11,
 'FONG, Tsz Ho': 11,
 'NG, Ka Man': 11,
 'BAO, Zhigang': 11,
 'ARYA, Sunil': 11,
 'WONG, Raymond C W': 11,
 'NASON, Stephen William': 11,
 'WONG, James K.': 11,
 'IP, Ivan Chi Ho': 11,
 'CHAN, Gary Shueng Han': 11,
 'SHEN, Yiwen': 11,
 'LEUNG, Chi Man': 11,
 'WANG, Xuan': 11,
 'JEONG, Martha': 11,
 'LIEM Rhea P': 11,
 'WONG, Tsz Wai': 11,
 'AU, Pak Hung': 11,
 'LAW, Kam Tuen': 11,
 'TANG, Rui': 11,
 'KAFSHDAR GOHARSHADY, Amir': 11,
 'WONG, Simon Man Ho': 11,
 'SHEONG, Frederick Fu Kit': 11,
 'ROSSITER, David': 11,
 'CHENG, Kam Hang': 11,
 'HUANG, Yong': 11,
 'ENGLAND, Graham James Fisher': 11,
 'MENG, Zili': 11,
 'SIU, Wai Sze Grace': 11,
 'SHIOMI, Koji': 11,
 'KAILA, Ilari Julius': 11,
 'HUA, Xinyu': 11,
 'SIU, Kam Wing': 11,
 'SHIEH, Tony': 11,
 'CHEUNG, Kwok Yip': 11,
 'FU, Li-tsui': 11,
 'SHI, Ling': 11,
 'DALTON, Amy N': 11,
 'LEUNG, Shing Yu': 11,
 'FRANKLIN, Laurence Craig': 11,
 'ZHANG, Hongtao': 11,
 'LI, Larry': 11,
 'PARK, Eunyoung': 11,
 'YU, Yan': 11

In [76]:
# helper function to extract course instructor name from the raw instructor column, which can be a string or a list of dicts
def parse_first_instructor_name(x):
    if pd.isna(x):
        return np.nan

    try:
        value = x

        if isinstance(value, str):
            value = value.strip()
            if not value or value == "[]":
                return np.nan
            value = ast.literal_eval(value)

        if isinstance(value, list) and len(value) > 0:
            first_item = value[0]
            if isinstance(first_item, dict):
                name = first_item.get("name")
                if pd.notna(name) and str(name).strip():
                    return str(name).strip()
            elif isinstance(first_item, str):
                return first_item.strip() or np.nan

        return np.nan
    except Exception:
        return np.nan

In [77]:
# helper function to extract course level from course name 
def get_level(course_name):
    # COMP1021 -> 1000
    try:
        d = str(course_name)[4]
        if d.isdigit():
            return int(d) * 1000
    except Exception:
        pass
    return np.nan

In [78]:
# cleaning for a single course file, return cleaned df and stats
TARGET_COLS = [
    "rating_workload",'rating_content', 'rating_teaching', 'rating_grading',
    "has_midterm", "has_final", "has_quiz", "has_assignment", "has_essay",
    "has_project", "has_attendance", "has_reading", "has_presentation",
    "course_name", "level", "upvote_ratio", "insturctor_rating"
]
def clean_one_course_file(
    fp,
    name_to_score,
    min_year=2020,
    max_year=2026,
    output_root=None,
    max_reviews_per_course=50
):
    course_name = fp.stem.replace("_reviews", "")
    df = pd.read_csv(fp, encoding="utf-8-sig")
    input_rows = len(df)

    df["course_name"] = course_name
    df["level"] = get_level(course_name)

    df["upvote_count"] = pd.to_numeric(df.get("upvote_count"), errors="coerce")
    df["vote_count"] = pd.to_numeric(df.get("vote_count"), errors="coerce")
    df["upvote_ratio"] = np.where(
        df["vote_count"].fillna(0) > 0,
        df["upvote_count"].fillna(0) / df["vote_count"],
        np.nan
    )

    df["insturctor_name"] = df.get(
        "instructors",
        pd.Series(index=df.index, dtype=object)
    ).apply(parse_first_instructor_name)

    before_drop_inst = len(df)
    df = df[df["insturctor_name"].notna()].copy()
    dropped_empty_instructors = before_drop_inst - len(df)

    df["insturctor_rating"] = df["insturctor_name"].map(name_to_score)

    df["date_parsed"] = pd.to_datetime(df.get("date"), errors="coerce")
    before_drop_year = len(df)
    df = df[df["date_parsed"].dt.year.between(min_year, max_year, inclusive="both")].copy()
    dropped_out_of_year_range = before_drop_year - len(df)

    # Keep at most N most recent reviews per course
    before_cap = len(df)
    df = df.sort_values("date_parsed", ascending=False).head(max_reviews_per_course).copy()
    dropped_by_cap = before_cap - len(df)

    for c in TARGET_COLS:
        if c not in df.columns:
            df[c] = np.nan
    df_out = df[TARGET_COLS].copy()

    if output_root is not None:
        course_out_dir = output_root / course_name
        course_out_dir.mkdir(parents=True, exist_ok=True)
        out_path = course_out_dir / f"{course_name}_cleaned.csv"
        df_out.to_csv(out_path, index=False, encoding="utf-8-sig")

    stats = {
        "course_name": course_name,
        "input_rows": input_rows,
        "dropped_empty_instructors": dropped_empty_instructors,
        "dropped_out_of_year_range": dropped_out_of_year_range,
        "dropped_by_cap": dropped_by_cap,
        "output_rows": len(df_out)
    }
    return df_out, stats

In [79]:
# test with comp1021 
MAX_REVIEWS_PER_COURSE = 50
max_reviews_per_course=MAX_REVIEWS_PER_COURSE

fp = ROOT / "COMP1021" / "COMP1021_reviews.csv"
out_dir = Path(r"C:\Users\aazh0\Desktop\IEDA3560_Project\ust_review_cleanned_baseline")
out_dir.mkdir(parents=True, exist_ok=True)

df_comp1021_cleaned, stats_comp1021 = clean_one_course_file(
    fp=fp,
    name_to_score=name_to_score,
    min_year=2020,
    max_year=2026,
    output_root=out_dir
)

df_comp1021_cleaned.head()

,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,has_attendance,has_reading,has_presentation,course_name,level,upvote_ratio,insturctor_rating
115,5,5,5,5,True,True,False,False,False,False,False,False,False,COMP1021,1000,NaN,11.0
116,4,3,4,3,True,True,False,True,False,False,False,False,False,COMP1021,1000,NaN,11.0
27,5,4,3,5,True,True,False,True,False,False,False,False,False,COMP1021,1000,1.0,NaN
117,5,5,5,5,True,True,False,True,False,False,False,False,False,COMP1021,1000,NaN,11.0
28,2,3,4,1,True,True,False,True,False,False,False,False,True,COMP1021,1000,1.0,NaN


In [80]:
stats_comp1021

{'course_name': 'COMP1021',
 'input_rows': 1234,
 'dropped_empty_instructors': 190,
 'dropped_out_of_year_range': 211,
 'dropped_by_cap': 783,
 'output_rows': 50}

In [81]:
def clean_all_courses(root=ROOT, name_to_score=None, min_year=MIN_YEAR, max_year=MAX_YEAR, output_root=OUTPUT_ROOT):
    files = sorted(root.glob("COMP*/COMP*_reviews.csv"))
    if not files:
        raise FileNotFoundError("No course review csv found under ./ust_reviews")
    if name_to_score is None:
        raise ValueError("name_to_score is required")

    all_cleaned = []
    report_rows = []

    for fp in files:
        df_out, stats = clean_one_course_file(
            fp=fp,
            name_to_score=name_to_score,
            min_year=min_year,
            max_year=max_year,
            output_root=output_root
        )
        all_cleaned.append(df_out)
        report_rows.append(stats)

    df_all_cleaned = pd.concat(all_cleaned, ignore_index=True)
    report_df = pd.DataFrame(report_rows).sort_values("output_rows", ascending=False)

    df_all_cleaned.to_csv(output_root / "all_courses_cleaned.csv", index=False, encoding="utf-8-sig")
    report_df.to_csv(output_root / "cleaning_report.csv", index=False, encoding="utf-8-sig")

    return df_all_cleaned, report_df

In [82]:
MAX_REVIEWS_PER_COURSE = 50
max_reviews_per_course=MAX_REVIEWS_PER_COURSE
ROOT = Path("./ust_reviews")  # input: raw data reviews
OUTPUT_ROOT = Path(r"C:\Users\aazh0\Desktop\IEDA3560_Project\ust_review_cleanned_baseline")  # output: 清洗後資料
RANKING_PATH = ROOT / "instructor_ranking.csv"

MIN_YEAR = 2020
MAX_YEAR = 2026

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [83]:
df_all_cleaned, report_df = clean_all_courses(
    root=ROOT,
    name_to_score=name_to_score,
    min_year=MIN_YEAR,
    max_year=MAX_YEAR,
    output_root=OUTPUT_ROOT
)

total_input = report_df["input_rows"].sum()
total_drop_inst = report_df["dropped_empty_instructors"].sum()
total_drop_year = report_df["dropped_out_of_year_range"].sum()
total_output = report_df["output_rows"].sum()
total_drop_cap = report_df["dropped_by_cap"].sum()


print("==== Cleaning Summary ====")
print("Total input rows:", total_input)
print("Dropped due to empty instructors:", total_drop_inst)
print(f"Dropped due to year filter [{MIN_YEAR}, {MAX_YEAR}]:", total_drop_year)
print(f"Dropped due to per-course cap ({MAX_REVIEWS_PER_COURSE}):", total_drop_cap)
print("Total output rows:", total_output)
print("Retention ratio:", f"{(total_output / total_input):.2%}" if total_input > 0 else "N/A")

display(report_df.head(20))
display(df_all_cleaned.head(20))

==== Cleaning Summary ====
Total input rows: 3904
Dropped due to empty instructors: 436
Dropped due to year filter [2020, 2026]: 1123
Dropped due to per-course cap (50): 1047
Total output rows: 1298
Retention ratio: 33.25%


,course_name,input_rows,dropped_empty_instructors,dropped_out_of_year_range,dropped_by_cap,output_rows
14,COMP2011,311,43,85,133,50
2,COMP1022P,356,40,224,42,50
15,COMP2012,109,8,36,15,50
4,COMP1023,63,7,0,6,50
1,COMP1021,1234,190,211,783,50
32,COMP3711,73,7,14,2,50
17,COMP2211,60,3,0,7,50
21,COMP2711H,66,0,13,3,50
20,COMP2711,177,19,52,56,50
16,COMP2012H,75,2,25,0,48


,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,has_attendance,has_reading,has_presentation,course_name,level,upvote_ratio,insturctor_rating
0,4,5,4,4,False,True,True,False,False,True,True,False,False,COMP1001,1000,NaN,0.0
1,3,3,3,3,True,True,False,False,False,True,False,False,False,COMP1001,1000,NaN,0.0
2,5,5,4,5,False,True,True,False,False,True,True,True,True,COMP1001,1000,NaN,0.0
3,5,5,5,5,False,True,True,False,False,True,True,False,True,COMP1001,1000,NaN,0.0
4,4,5,5,5,False,True,True,True,False,True,True,False,True,COMP1001,1000,NaN,0.0
5,4,4,5,5,False,True,False,True,False,True,True,False,False,COMP1001,1000,NaN,0.0
6,4,4,5,4,False,True,True,False,False,True,False,False,False,COMP1001,1000,NaN,0.0
7,5,4,5,5,False,True,True,True,False,True,True,False,True,COMP1001,1000,0.500000,0.0
8,5,4,4,5,False,True,False,True,False,True,True,False,False,COMP1001,1000,0.500000,0.0
9,4,5,4,5,False,True,False,True,False,True,True,False,False,COMP1001,1000,0.000000,0.0


In [84]:
report_df.head(20)

,course_name,input_rows,dropped_empty_instructors,dropped_out_of_year_range,dropped_by_cap,output_rows
14,COMP2011,311,43,85,133,50
2,COMP1022P,356,40,224,42,50
15,COMP2012,109,8,36,15,50
4,COMP1023,63,7,0,6,50
1,COMP1021,1234,190,211,783,50
32,COMP3711,73,7,14,2,50
17,COMP2211,60,3,0,7,50
21,COMP2711H,66,0,13,3,50
20,COMP2711,177,19,52,56,50
16,COMP2012H,75,2,25,0,48


In [85]:
df_all_cleaned.head()

,rating_workload,rating_content,rating_teaching,rating_grading,has_midterm,has_final,has_quiz,has_assignment,has_essay,has_project,has_attendance,has_reading,has_presentation,course_name,level,upvote_ratio,insturctor_rating
0,4,5,4,4,False,True,True,False,False,True,True,False,False,COMP1001,1000,NaN,0.0
1,3,3,3,3,True,True,False,False,False,True,False,False,False,COMP1001,1000,NaN,0.0
2,5,5,4,5,False,True,True,False,False,True,True,True,True,COMP1001,1000,NaN,0.0
3,5,5,5,5,False,True,True,False,False,True,True,False,True,COMP1001,1000,NaN,0.0
4,4,5,5,5,False,True,True,True,False,True,True,False,True,COMP1001,1000,NaN,0.0


In [86]:
total_rows = len(df_all_cleaned)
nan_upvote_ratio_count = df_all_cleaned["upvote_ratio"].isna().sum()

print("Total rows:", total_rows)
print("upvote_ratio is NaN:", nan_upvote_ratio_count)

Total rows: 1298
upvote_ratio is NaN: 471


In [87]:
level1_df = df_all_cleaned[df_all_cleaned["level"] == 1000].copy()
print("Level 1000 rows:", len(level1_df))
level2_df = df_all_cleaned[df_all_cleaned["level"] == 2000].copy()
print("Level 2000 rows:", len(level2_df))
level3_df = df_all_cleaned[df_all_cleaned["level"] == 3000].copy()
print("Level 3000 rows:", len(level3_df))
level4_df = df_all_cleaned[df_all_cleaned["level"] == 4000].copy()
print("Level 4000 rows:", len(level4_df))
level5_df = df_all_cleaned[df_all_cleaned["level"] == 5000].copy()
level5_df.head()
print("Level 5000 rows:", len(level5_df))

Level 1000 rows: 368
Level 2000 rows: 342
Level 3000 rows: 285
Level 4000 rows: 221
Level 5000 rows: 82


In [88]:
# check review distribution over the course level 
level_course_counts = (
    df_all_cleaned
    .groupby(["level", "course_name"], as_index=False)
    .size()
    .rename(columns={"size": "review_count"})
    .sort_values(["level", "review_count"], ascending=[True, False])
)

level_course_counts[level_course_counts["level"] == 1000] 

,level,course_name,review_count
1,1000,COMP1021,50
2,1000,COMP1022P,50
4,1000,COMP1023,50
10,1000,COMP1942,44
3,1000,COMP1022Q,38
0,1000,COMP1001,37
12,1000,COMP1944,29
6,1000,COMP1029C,17
7,1000,COMP1029J,15
9,1000,COMP1029V,15


In [89]:
level_course_counts[level_course_counts["level"] == 2000]

,level,course_name,review_count
13,2000,COMP2011,50
14,2000,COMP2012,50
16,2000,COMP2211,50
19,2000,COMP2711,50
20,2000,COMP2711H,50
15,2000,COMP2012H,48
17,2000,COMP2611,36
18,2000,COMP2633,8


In [51]:
level_course_counts[level_course_counts["level"] == 3000]

,level,course_name,review_count
31,3000,COMP3711,52
27,3000,COMP3511,48
21,3000,COMP3021,38
23,3000,COMP3111,32
26,3000,COMP3311,27
22,3000,COMP3031,24
33,3000,COMP3721,18
25,3000,COMP3211,14
32,3000,COMP3711H,14
29,3000,COMP3632,8


In [90]:
level_course_counts[level_course_counts["level"] == 4000]

,level,course_name,review_count
36,4000,COMP4211,22
45,4000,COMP4441,17
34,4000,COMP4021,13
43,4000,COMP4421,12
44,4000,COMP4431,11
41,4000,COMP4332,9
49,4000,COMP4471,9
61,4000,COMP4651,9
42,4000,COMP4411,8
50,4000,COMP4521,8


In [53]:
level_course_counts[level_course_counts["level"] == 5000]

,level,course_name,review_count
70,5000,COMP5212,36
73,5000,COMP5621,22
69,5000,COMP5211,13
74,5000,COMP5631,6
72,5000,COMP5311,4
71,5000,COMP5221,1
